## Data Cleaning, KPI Engineering & Feature Creation



### Phase 1 :Data Cleaning & Standardization

Loaded the scraped military dataset

In [1]:
import pandas as pd
df = pd.read_csv("/content/Military_raw_data (1).csv")

The dataset was successfully loaded.


Initial Data Inspection

In [2]:
print(df.head())
print(df.info())

                        country  power_index_rank  power_index_score  \
0   2026Benin Military Strength               137             3.8963   
1  2026Mexico Military Strength                36             0.6401   
2   2026Kenya Military Strength                84             1.8588   
3   2026Japan Military Strength                 7             0.1876   
4  2026Bhutan Military Strength               145             5.7991   

   purchasing_power_parity  foreign_exchange/gold  defense_budget  \
0                      116                    134             137   
1                       13                     18              39   
2                       58                     78              81   
3                        5                      2               9   
4                      140                    130             143   

   external_debt  square_land_area  coastline_coverage  shared_borders  ...  \
0             33                96                   8              52  .

Dataset contains 145 countries and 55 columns
Several numeric columns are stored as object (text) type

### Identify Data Noise

In [3]:
df.columns.tolist()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 66 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   country                        146 non-null    object 
 1   power_index_rank               146 non-null    int64  
 2   power_index_score              146 non-null    float64
 3   purchasing_power_parity        146 non-null    int64  
 4   foreign_exchange/gold          146 non-null    int64  
 5   defense_budget                 146 non-null    int64  
 6   external_debt                  146 non-null    int64  
 7   square_land_area               146 non-null    int64  
 8   coastline_coverage             146 non-null    int64  
 9   shared_borders                 146 non-null    int64  
 10  waterways_(usable)             146 non-null    int64  
 11  total_population               146 non-null    int64  
 12  available_manpower             146 non-null    int

### Clean Numeric Columns

In [4]:
def clean_numeric(col):
    return (
        col.astype(str)
           .str.replace(r'[^0-9.]', '', regex=True)
           .replace('', '0')
           .astype(float)
    )

for col in df.columns:
    if col != 'country':
        df[col] = clean_numeric(df[col]
                                )

All numeric values converted to usable numeric format

### Standardize Column Names

Removing extra spaces

In [5]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

### Handle Missing Values

In [6]:
df = df.fillna(0)

Validate Cleaned Data

In [7]:
df.info()
df.describe()
df.isnull().sum()
df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 66 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   country                        146 non-null    object 
 1   power_index_rank               146 non-null    float64
 2   power_index_score              146 non-null    float64
 3   purchasing_power_parity        146 non-null    float64
 4   foreign_exchange/gold          146 non-null    float64
 5   defense_budget                 146 non-null    float64
 6   external_debt                  146 non-null    float64
 7   square_land_area               146 non-null    float64
 8   coastline_coverage             146 non-null    float64
 9   shared_borders                 146 non-null    float64
 10  waterways_(usable)             146 non-null    float64
 11  total_population               146 non-null    float64
 12  available_manpower             146 non-null    flo

np.int64(1)

### Export Cleaned Dataset

In [8]:
df.to_csv("military_data_cleaned.csv", index=False)

## Phase 2: KPI Engineering & Feature Creation

Load Clean Dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("military_data_cleaned.csv")
print(df.head())

### Assets per Capita

In [16]:
import numpy as np
df["assets_per_capita"] = (
    df["aircraft_total"] +
    df["tanks"] +
    df["navy_personnel*"]
) / df["available_manpower"].replace(0, np.nan)

Budget-to-GDP Ratio

In [18]:
df["budget_to_gdp_ratio"] = (
    df["defense_budget"] /
    df["purchasing_power_parity"].replace(0, np.nan)
)

Power Index Rank Gap

In [19]:
df["rank"] = df["defense_budget"].rank(ascending=False)

top_rank = df["rank"].min()

df["power_index_rank"] = df["rank"] - top_rank

Strength Category

In [20]:
df["strength_category"] = pd.cut(
    df["assets_per_capita"],
    bins=3,
    labels=["Low", "Medium", "High"]
)

Validate KPI Values

In [21]:
import numpy as np

print(np.isinf(df.select_dtypes(include=[float, int])).sum())

power_index_rank           0
power_index_score          0
purchasing_power_parity    0
foreign_exchange/gold      0
defense_budget             0
                          ..
railway_coverage           0
total_tonnage              0
assets_per_capita          0
budget_to_gdp_ratio        0
rank                       0
Length: 68, dtype: int64


Verify KPI Distribution

In [23]:
print(df[[
    "assets_per_capita",
    "budget_to_gdp_ratio",
    "power_index_rank"
]].describe())

       assets_per_capita  budget_to_gdp_ratio  power_index_rank
count         146.000000           146.000000         146.00000
mean            3.848716             1.114705          72.50000
std             3.482989             0.587121          42.29062
min             0.473684             0.304348           0.00000
25%             2.312677             0.778388          36.25000
50%             2.862782             0.980324          72.50000
75%             4.241026             1.266106         108.75000
max            27.000000             4.125000         145.00000


Manual Sample Validation

In [25]:
print(df[[
    "country",
    "assets_per_capita",
    "budget_to_gdp_ratio",
    "power_index_rank"
]].head(5))

                        country  assets_per_capita  budget_to_gdp_ratio  \
0   2026Benin Military Strength           4.350649             1.181034   
1  2026Mexico Military Strength          18.000000             3.000000   
2   2026Kenya Military Strength           5.870968             1.396552   
3   2026Japan Military Strength           3.538462             1.800000   
4  2026Bhutan Military Strength           2.857143             1.021429   

   power_index_rank  
0               8.0  
1             106.0  
2              64.0  
3             136.0  
4               2.0  


### Export Final Dataset

In [26]:
df.to_csv("military_final.csv", index=False)